In [1]:
# Cell 1: Installation (Unpinned SentencePiece)
!pip install -q transformers==4.38.2 \
                sentencepiece \
                evaluate==0.4.1 \
                jedi

!pip check

gradio 6.31.0 has requirement huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2.
gradio-client 2.7.2 has requirement huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2.
sentence-transformers 5.7.0 has requirement tokenizers>=0.19, but you have tokenizers 0.15.2.
sentence-transformers 5.7.0 has requirement transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2.
diffusers 0.42.1 has requirement huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2.


In [2]:
import torch
import numpy as np
import random
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set working directory to your project root
project_path = '/content/drive/MyDrive/ATR/ESM_Research_Project'
os.chdir(project_path)
print(f"Working Directory set to: {os.getcwd()}")

# 3. System Diagnostics
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Available: {gpu_available}")
if gpu_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
else:
    print("WARNING: Runtime is on CPU. Go to Runtime > Change runtime type > T4 GPU.")

# 4. Deterministic Seed Control
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Random seeds locked to 42.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working Directory set to: /content/drive/MyDrive/ATR/ESM_Research_Project
Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device: Tesla T4
VRAM Allocated: 0.00 GB
Random seeds locked to 42.


In [3]:
# Cell 3: Executable Environment Test (Fixed Tokenizer)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

def test_environment():
    print("Initiating model download and VRAM allocation...")
    model_name = "cross-encoder/nli-deberta-v3-large"

    try:
        # Added use_fast=False to bypass the Rust tokenizer bug
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        print("Tokenizer loaded successfully.")

        model = AutoModelForSequenceClassification.from_pretrained(model_name).to("cuda")
        model.eval()
        print("Model weights loaded to CUDA successfully.")

        # Dummy Forward Pass
        premise = "The treatment is associated with recovery."
        hypothesis = "The treatment causes recovery."

        inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)

        print(f"Forward pass successful. Logit shape: {outputs.logits.shape}")
        print("ENVIRONMENT TEST PASSED.")

    except Exception as e:
        print(f"ENVIRONMENT TEST FAILED: {str(e)}")

test_environment()

Initiating model download and VRAM allocation...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded successfully.


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Model weights loaded to CUDA successfully.
Forward pass successful. Logit shape: torch.Size([1, 3])
ENVIRONMENT TEST PASSED.


In [18]:
import pandas as pd
import os

def load_and_prepare_data(filepath):
    print(f"Loading data from {filepath}...")
    df = pd.read_csv(filepath)

    # 1. Schema Validation
    required_cols = {'pair_id', 'condition', 'text'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"Missing columns. Expected {required_cols}, found {set(df.columns)}")

    processed_rows = []
    skipped_pairs = 0

    # 2. Grouping and Pair Generation
    for pair_id, group in df.groupby("pair_id"):
        # Ensure exact triplet structure
        if len(group) != 3:
            skipped_pairs += 1
            continue

        orig = group[group["condition"] == "original"]["text"].values
        para = group[group["condition"] == "paraphrase"]["text"].values
        esc = group[group["condition"] == "escalated"]["text"].values

        # Ensure exactly one of each condition exists
        if len(orig) != 1 or len(para) != 1 or len(esc) != 1:
            skipped_pairs += 1
            continue

        orig_text, para_text, esc_text = orig[0], para[0], esc[0]

        # Pair 1: Negative Control (Paraphrase)
        processed_rows.append({
            "pair_id": pair_id,
            "test_condition": "paraphrase",
            "reference_text": orig_text,
            "hypothesis_text": para_text,
            "ground_truth_label": 0
        })

        # Pair 2: Test Case (Escalation)
        processed_rows.append({
            "pair_id": pair_id,
            "test_condition": "escalated",
            "reference_text": orig_text,
            "hypothesis_text": esc_text,
            "ground_truth_label": 1
        })

    processed_df = pd.DataFrame(processed_rows)
    print(f"Successfully processed {len(processed_df) // 2} complete triplets.")
    if skipped_pairs > 0:
        print(f"WARNING: Skipped {skipped_pairs} pairs due to malformed conditions.")

    return processed_df

# --- Sanity/Unit Test ---
if __name__ == "__main__":
    file_name = "/content/drive/MyDrive/ATR/ESM_Research_Project/data/custom_epistemic_suite.csv"

    # Optional: Create a dummy file if the real one isn't in the current dir for the test to run
    if not os.path.exists(file_name):
        print(f"Dummy creation: {file_name} not found, generating a synthetic one for the test.")
        pd.DataFrame([
            {"pair_id": "P001", "condition": "original", "text": "A is associated with B."},
            {"pair_id": "P001", "condition": "paraphrase", "text": "A is correlated with B."},
            {"pair_id": "P001", "condition": "escalated", "text": "A causes B."}
        ]).to_csv(file_name, index=False)

    try:
        eval_df = load_and_prepare_data(file_name)
        print("\n--- Output Shapes and Types ---")
        print(f"DataFrame Shape: {eval_df.shape} (Expected: N*2 rows, 5 columns)")
        print("\n--- First 2 Rows Preview ---")
        print(eval_df.head(2).to_string())
    except Exception as e:
        print(f"TEST FAILED: {str(e)}")

Loading data from /content/drive/MyDrive/ATR/ESM_Research_Project/data/custom_epistemic_suite.csv...
Successfully processed 100 complete triplets.

--- Output Shapes and Types ---
DataFrame Shape: (200, 5) (Expected: N*2 rows, 5 columns)

--- First 2 Rows Preview ---
  pair_id test_condition                                                                                                                                                                                                                                             reference_text                                                                                                                                                                                                                                               hypothesis_text  ground_truth_label
0    P001     paraphrase  Our findings suggest that while high-dimensional signatures offer extensive predictive power, the consensus-based core genes - particularly TCF7L2 - represe

In [19]:
!pip install -q rouge_score

import pandas as pd
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Ensure NLTK tokenizer is available
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def compute_lexical_baselines_native(df):
    print("Initializing native BLEU and ROUGE scorers...")
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    chencherry = SmoothingFunction().method1

    preds = df["hypothesis_text"].tolist()
    refs = df["reference_text"].tolist()

    rouge_scores = []
    bleu_scores = []

    print("Computing sentence-level ROUGE-L and BLEU...")
    for hyp, ref in zip(preds, refs):
        # 1. ROUGE-L (F-measure)
        r_score = rouge.score(ref, hyp)['rougeL'].fmeasure
        rouge_scores.append(r_score)

        # 2. BLEU
        ref_tokens = nltk.word_tokenize(ref)
        hyp_tokens = nltk.word_tokenize(hyp)
        # Smoothing prevents 0.0 scores on short sentences missing 4-grams
        b_score = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=chencherry)
        bleu_scores.append(b_score)

    df["score_rougeL"] = rouge_scores
    df["score_bleu"] = bleu_scores
    return df

# --- Sanity/Unit Test ---
if __name__ == "__main__":
    try:
        eval_df = compute_lexical_baselines_native(eval_df)
        print("\n--- Output Shapes and Types ---")
        print(f"DataFrame Shape: {eval_df.shape} (Expected: N*2 rows, 7 columns)")
        print("\n--- First 2 Rows Preview (Lexical Scores) ---")
        print(eval_df[['test_condition', 'hypothesis_text', 'score_rougeL', 'score_bleu']].head(2).to_string())
    except Exception as e:
        print(f"TEST FAILED: {str(e)}")

Initializing native BLEU and ROUGE scorers...
Computing sentence-level ROUGE-L and BLEU...

--- Output Shapes and Types ---
DataFrame Shape: (200, 7) (Expected: N*2 rows, 7 columns)

--- First 2 Rows Preview (Lexical Scores) ---
  test_condition                                                                                                                                                                                                                                               hypothesis_text  score_rougeL  score_bleu
0     paraphrase     Our findings suggest that while high-dimensional signatures offer extensive predictive power, the consensus-based core genes - particularly TCF7L2 - represent reproducible candidate biomarkers correlated with disease progression and patient outcome.      0.966667    0.915710
1      escalated  Our findings suggest that while high-dimensional signatures offer extensive predictive power, the consensus-based core genes - particularly TCF7L2 - represent 

In [20]:
!pip install -q bert-score

import pandas as pd
from bert_score import score
import torch
import gc

def compute_semantic_baseline(df):
    print("Initializing BERTScore (microsoft/deberta-xlarge-mnli)...")
    preds = df["hypothesis_text"].tolist()
    refs = df["reference_text"].tolist()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Compute BERTScore (returns Precision, Recall, and F1 tensors)
    # We extract F1 as our primary semantic similarity metric
    P, R, F1 = score(preds, refs, model_type="microsoft/deberta-xlarge-mnli", lang="en", verbose=True, device=device)

    df["score_bertscore"] = F1.cpu().numpy()

    # Flush GPU VRAM to prepare for the final NLI metric
    torch.cuda.empty_cache()
    gc.collect()

    return df

# --- Sanity/Unit Test ---
if __name__ == "__main__":
    try:
        # Assumes 'eval_df' is still in memory from Cycle 2
        eval_df = compute_semantic_baseline(eval_df)
        print("\n--- Output Shapes and Types ---")
        print(f"DataFrame Shape: {eval_df.shape} (Expected: N*2 rows, 8 columns)")
        print("\n--- First 2 Rows Preview (Semantic Score Added) ---")
        print(eval_df[['test_condition', 'hypothesis_text', 'score_bertscore']].head(2).to_string())
    except Exception as e:
        print(f"TEST FAILED: {str(e)}")

Initializing BERTScore (microsoft/deberta-xlarge-mnli)...
calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 7.58 seconds, 26.37 sentences/sec

--- Output Shapes and Types ---
DataFrame Shape: (200, 8) (Expected: N*2 rows, 8 columns)

--- First 2 Rows Preview (Semantic Score Added) ---
  test_condition                                                                                                                                                                                                                                               hypothesis_text  score_bertscore
0     paraphrase     Our findings suggest that while high-dimensional signatures offer extensive predictive power, the consensus-based core genes - particularly TCF7L2 - represent reproducible candidate biomarkers correlated with disease progression and patient outcome.         0.997469
1      escalated  Our findings suggest that while high-dimensional signatures offer extensive predictive power, the consensus-based core genes - particularly TCF7L2 - represent reproducible candidate biomarkers that contribute to disease 

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import gc

def compute_epistemic_shift(df):
    print("Loading Asymmetrical NLI Model (cross-encoder/nli-deberta-v3-large)...")
    model_name = "cross-encoder/nli-deberta-v3-large"

    # use_fast=False bypasses PyPreTokenizerTypeWrapper bug
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    def get_entailment_prob(premise, hypothesis):
        inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        # DeBERTa-v3 MNLI mapping: 0=Contradiction, 1=Entailment, 2=Neutral
        probs = F.softmax(logits, dim=1)
        return probs[0][1].item()

    esm_scores = []

    print("Computing Forward and Backward Entailments...")
    for idx, row in df.iterrows():
        s_orig = row["reference_text"]
        s_edit = row["hypothesis_text"]

        # P(S_orig | S_edit) -> Premise = edit, Hypothesis = orig
        p_orig_given_edit = get_entailment_prob(premise=s_edit, hypothesis=s_orig)

        # P(S_edit | S_orig) -> Premise = orig, Hypothesis = edit
        p_edit_given_orig = get_entailment_prob(premise=s_orig, hypothesis=s_edit)

        # Delta = P(orig | edit) - P(edit | orig)
        delta_epistemic = p_orig_given_edit - p_edit_given_orig
        esm_scores.append(delta_epistemic)

    df["score_delta_epistemic"] = esm_scores

    # Flush GPU VRAM to ensure we have memory for plotting/meta-evaluation next
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    return df

# --- Sanity/Unit Test ---
if __name__ == "__main__":
    try:
        # Assumes 'eval_df' is still in memory from Cycle 3
        eval_df = compute_epistemic_shift(eval_df)
        print("\n--- Output Shapes and Types ---")
        print(f"DataFrame Shape: {eval_df.shape} (Expected: N*2 rows, 9 columns)")
        print("\n--- First 2 Rows Preview (ESM Added) ---")
        # Selecting key columns to verify metric separation
        print(eval_df[['test_condition', 'hypothesis_text', 'score_bertscore', 'score_delta_epistemic']].head(2).to_string())
    except Exception as e:
        print(f"TEST FAILED: {str(e)}")

Loading Asymmetrical NLI Model (cross-encoder/nli-deberta-v3-large)...
Computing Forward and Backward Entailments...

--- Output Shapes and Types ---
DataFrame Shape: (200, 9) (Expected: N*2 rows, 9 columns)

--- First 2 Rows Preview (ESM Added) ---
  test_condition                                                                                                                                                                                                                                               hypothesis_text  score_bertscore  score_delta_epistemic
0     paraphrase     Our findings suggest that while high-dimensional signatures offer extensive predictive power, the consensus-based core genes - particularly TCF7L2 - represent reproducible candidate biomarkers correlated with disease progression and patient outcome.         0.997469               0.000096
1      escalated  Our findings suggest that while high-dimensional signatures offer extensive predictive power, the consensus-ba

In [22]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def compute_meta_evaluation_and_plot(df):
    print("Preparing output directories...")
    os.makedirs("results/Figures", exist_ok=True)

    y_true = df["ground_truth_label"].values

    # Align metrics for ROC-AUC (Similarity metrics inverted)
    metrics = {
        "Δ_epistemic": df["score_delta_epistemic"].values,
        "BERTScore": -df["score_bertscore"].values,
        "ROUGE-L": -df["score_rougeL"].values,
        "BLEU": -df["score_bleu"].values
    }

    # 1. Statistical Calculations (Table 1)
    def get_bootstrap_ci(y_t, y_s, n_bootstraps=2000): # 2k for speed in test, use 10k for paper
        rng = np.random.RandomState(42)
        bootstrapped_scores = []
        for _ in range(n_bootstraps):
            indices = rng.randint(0, len(y_s), len(y_s))
            if len(np.unique(y_t[indices])) < 2: continue
            score = roc_auc_score(y_t[indices], y_s[indices])
            bootstrapped_scores.append(score)
        return np.percentile(bootstrapped_scores, 2.5), np.percentile(bootstrapped_scores, 97.5)

    print("\n" + "="*70)
    print(f"{'Metric':<15} | {'ROC-AUC':<10} | {'95% CI':<20} | {'Spearman r'}")
    print("-" * 70)

    for name, scores in metrics.items():
        auc = roc_auc_score(y_true, scores)
        ci_lower, ci_upper = get_bootstrap_ci(y_true, scores)

        # Spearman uses original un-inverted scores to show raw correlation direction
        raw_scores = scores if name == "Δ_epistemic" else -scores
        rho, _ = spearmanr(raw_scores, y_true)

        print(f"{name:<15} | {auc:<10.4f} | [{ci_lower:.3f}, {ci_upper:.3f}]       | {rho:.4f}")
    print("="*70)

    # 2. Figure 1: ROC Curves
    plt.figure(figsize=(8, 6))
    for name, scores in metrics.items():
        fpr, tpr, _ = roc_curve(y_true, scores)
        auc = roc_auc_score(y_true, scores)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Figure 1: ROC Curves for Epistemic Shift Detection")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.savefig("results/Figures/ROC_Curve.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 3. Figure 2: KDE Density Distribution
    plt.figure(figsize=(8, 5))
    sns.kdeplot(df[df["ground_truth_label"] == 0]["score_delta_epistemic"], label="Paraphrase (Control)", fill=True, color="blue")
    sns.kdeplot(df[df["ground_truth_label"] == 1]["score_delta_epistemic"], label="Escalated (Test)", fill=True, color="red")
    plt.xlabel("Δ_epistemic Score")
    plt.ylabel("Density")
    plt.title("Figure 2: Distribution of Epistemic Shifts")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig("results/Figures/KDE_Distribution.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 4. Save Final CSV
    output_cols = ["pair_id", "test_condition", "ground_truth_label", "score_bleu", "score_rougeL", "score_bertscore", "score_delta_epistemic"]
    final_df = df[output_cols]
    final_df.to_csv("results/esm_evaluation_results.csv", index=False)

    print("\nVisualizations saved to 'results/Figures/'.")
    print("Final output saved to 'results/esm_evaluation_results.csv'.")
    return final_df

# --- Sanity/Unit Test ---
if __name__ == "__main__":
    try:
        final_df = compute_meta_evaluation_and_plot(eval_df)
    except Exception as e:
        print(f"TEST FAILED: {str(e)}")

Preparing output directories...

Metric          | ROC-AUC    | 95% CI               | Spearman r
----------------------------------------------------------------------
Δ_epistemic     | 0.3234     | [0.247, 0.404]       | -0.3059
BERTScore       | 0.9518     | [0.918, 0.977]       | -0.7826
ROUGE-L         | 0.9152     | [0.868, 0.953]       | -0.7192
BLEU            | 0.8012     | [0.736, 0.860]       | -0.5217

Visualizations saved to 'results/Figures/'.
Final output saved to 'results/esm_evaluation_results.csv'.


In [23]:
# cycle 4(After Modification)
!pip install -q accelerate transformers

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc
import re

def compute_epistemic_shift_llm(df):
    print("Loading Generative NLI Model (Qwen/Qwen2.5-1.5B-Instruct in fp16)...")
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load in float16 to ensure it comfortably fits inside the Tesla T4 15GB VRAM
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
    model.eval()

    def get_entailment_prob(premise, hypothesis):
        # The targeted theoretical modification: explicit rule injection
        prompt = (
            f"Premise: '{premise}'\n"
            f"Hypothesis: '{hypothesis}'\n\n"
            "Rule 1: A correlational finding (e.g., 'associated with') does NOT entail a causal finding.\n"
            "Rule 2: A causal finding (e.g., 'causes', 'contributes to') logically ENTAILS a correlational finding.\n\n"
            "Task: Based strictly on these rules, what is the probability (from 0.0 to 1.0) that the Premise entails the Hypothesis?\n"
            "Output only the numeric float."
        )

        messages = [
            {"role": "system", "content": "You are a strict, logical biomedical text analyzer."},
            {"role": "user", "content": prompt}
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(device)

        # do_sample=False forces deterministic, greedy decoding matching your established seed constraints
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)

        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        # Regex to extract the raw float in case the LLM disobeys the "only float" formatting constraint
        match = re.search(r"0\.\d+|1\.0|0", response)
        if match:
            return float(match.group())
        return 0.5 # Neutral fallback if parsing fails

    esm_scores = []

    print("Computing Forward and Backward Entailments via LLM Generation...")
    for idx, row in df.iterrows():
        s_orig = row["reference_text"]
        s_edit = row["hypothesis_text"]

        # P(S_orig | S_edit) -> Premise = edit, Hypothesis = orig
        p_orig_given_edit = get_entailment_prob(premise=s_edit, hypothesis=s_orig)

        # P(S_edit | S_orig) -> Premise = orig, Hypothesis = edit
        p_edit_given_orig = get_entailment_prob(premise=s_orig, hypothesis=s_edit)

        # Delta = P(orig | edit) - P(edit | orig)
        delta_epistemic = p_orig_given_edit - p_edit_given_orig
        esm_scores.append(delta_epistemic)

    df["score_delta_epistemic_llm"] = esm_scores

    # Flush GPU VRAM
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    return df

# --- Sanity/Unit Test ---
if __name__ == "__main__":
    try:
        # Assuming eval_df is already in memory
        eval_df = compute_epistemic_shift_llm(eval_df)
        print("\n--- Output Shapes and Types ---")
        print(f"DataFrame Shape: {eval_df.shape}")
        print("\n--- First 2 Rows Preview (Generative ESM Added) ---")
        print(eval_df[['test_condition', 'score_bertscore', 'score_delta_epistemic', 'score_delta_epistemic_llm']].head(2).to_string())
    except Exception as e:
        print(f"TEST FAILED: {str(e)}")

Loading Generative NLI Model (Qwen/Qwen2.5-1.5B-Instruct in fp16)...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Computing Forward and Backward Entailments via LLM Generation...

--- Output Shapes and Types ---
DataFrame Shape: (200, 10)

--- First 2 Rows Preview (Generative ESM Added) ---
  test_condition  score_bertscore  score_delta_epistemic  score_delta_epistemic_llm
0     paraphrase         0.997469               0.000096                        0.4
1      escalated         0.988092              -0.001097                        0.0


In [24]:
#cycle 5(After Modification)
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Print the corrected sanity check
print("\n--- First 2 Rows Preview (Generative ESM Added) ---")
print(eval_df[['test_condition', 'score_bertscore', 'score_delta_epistemic_llm']].head(2).to_string())

# 2. Run Meta-Evaluation on the new LLM metric
print("\nPreparing output directories...")
os.makedirs("results/Figures", exist_ok=True)

y_true = eval_df["ground_truth_label"].values

# Align metrics for ROC-AUC (Similarity metrics inverted)
metrics = {
    "Δ_epistemic_LLM": eval_df["score_delta_epistemic_llm"].values,
    "BERTScore": -eval_df["score_bertscore"].values,
    "ROUGE-L": -eval_df["score_rougeL"].values,
    "BLEU": -eval_df["score_bleu"].values
}

def get_bootstrap_ci(y_t, y_s, n_bootstraps=2000):
    rng = np.random.RandomState(42)
    bootstrapped_scores = []
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_s), len(y_s))
        if len(np.unique(y_t[indices])) < 2: continue
        score = roc_auc_score(y_t[indices], y_s[indices])
        bootstrapped_scores.append(score)
    return np.percentile(bootstrapped_scores, 2.5), np.percentile(bootstrapped_scores, 97.5)

print("\n" + "="*70)
print(f"{'Metric':<17} | {'ROC-AUC':<10} | {'95% CI':<20} | {'Spearman r'}")
print("-" * 70)

for name, scores in metrics.items():
    auc = roc_auc_score(y_true, scores)
    ci_lower, ci_upper = get_bootstrap_ci(y_true, scores)

    raw_scores = scores if name == "Δ_epistemic_LLM" else -scores
    rho, _ = spearmanr(raw_scores, y_true)

    print(f"{name:<17} | {auc:<10.4f} | [{ci_lower:.3f}, {ci_upper:.3f}]       | {rho:.4f}")
print("="*70)

# Figure 1: ROC Curves
plt.figure(figsize=(8, 6))
for name, scores in metrics.items():
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Figure 1: ROC Curves for LLM Epistemic Shift Detection")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig("results/Figures/ROC_Curve_LLM.png", dpi=300, bbox_inches='tight')
plt.close()

# Figure 2: KDE Density Distribution
plt.figure(figsize=(8, 5))
sns.kdeplot(eval_df[eval_df["ground_truth_label"] == 0]["score_delta_epistemic_llm"], label="Paraphrase (Control)", fill=True, color="blue")
sns.kdeplot(eval_df[eval_df["ground_truth_label"] == 1]["score_delta_epistemic_llm"], label="Escalated (Test)", fill=True, color="red")
plt.xlabel("Δ_epistemic_LLM Score")
plt.ylabel("Density")
plt.title("Figure 2: Distribution of LLM Epistemic Shifts")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("results/Figures/KDE_Distribution_LLM.png", dpi=300, bbox_inches='tight')
plt.close()

output_cols = ["pair_id", "test_condition", "ground_truth_label", "score_bleu", "score_rougeL", "score_bertscore", "score_delta_epistemic_llm"]
eval_df[output_cols].to_csv("results/esm_evaluation_results_llm.csv", index=False)

print("\nVisualizations saved to 'results/Figures/'.")


--- First 2 Rows Preview (Generative ESM Added) ---
  test_condition  score_bertscore  score_delta_epistemic_llm
0     paraphrase         0.997469                        0.4
1      escalated         0.988092                        0.0

Preparing output directories...

Metric            | ROC-AUC    | 95% CI               | Spearman r
----------------------------------------------------------------------
Δ_epistemic_LLM   | 0.6002     | [0.545, 0.655]       | 0.2425
BERTScore         | 0.9518     | [0.918, 0.977]       | -0.7826
ROUGE-L           | 0.9152     | [0.868, 0.953]       | -0.7192
BLEU              | 0.8012     | [0.736, 0.860]       | -0.5217

Visualizations saved to 'results/Figures/'.


In [25]:
import torch
import torch.nn.functional as F
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import gc

# ---------------------------------------------------------
# Test 1: Discrete Classification Alignment
# ---------------------------------------------------------
def run_discrete_classification_alignment(df):
    print("--- Running Discrete Classification Test ---")
    print("Loading zero-shot classifier (facebook/bart-large-mnli)...")
    device_id = 0 if torch.cuda.is_available() else -1
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device_id)

    labels = ["direct causal", "conditional causal", "correlational", "no relationship"]
    discrepancies = 0

    # Test on the first 10 rows to save time
    for _, row in df.head(10).iterrows():
        orig_res = classifier(row['reference_text'], labels)
        hyp_res = classifier(row['hypothesis_text'], labels)

        orig_top = orig_res['labels'][0]
        hyp_top = hyp_res['labels'][0]

        # If the edit is an escalation, the label should shift from correlational -> causal
        if row['ground_truth_label'] == 1 and orig_top == hyp_top:
            discrepancies += 1

    print(f"Discrete Classification Failures on Escalations (First 10 pairs): {discrepancies}/10")

    del classifier
    torch.cuda.empty_cache()
    gc.collect()

# ---------------------------------------------------------
# Test 2: NLI Lexical Resilience
# ---------------------------------------------------------
def run_nli_lexical_resilience():
    print("\n--- Running NLI Lexical Resilience Test ---")
    print("Loading Asymmetrical NLI Model (cross-encoder/nli-deberta-v3-large)...")

    model_name = "cross-encoder/nli-deberta-v3-large"
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()

    def get_prob(premise, hypothesis):
        inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            probs = F.softmax(model(**inputs).logits, dim=1)
        return probs[0][1].item()

    premise = "The TCF7L2 gene is correlated with disease progression."

    # High overlap, explicit contradiction
    hypothesis_1 = "The TCF7L2 gene is not correlated with disease progression."
    # High overlap, causal escalation
    hypothesis_2 = "The TCF7L2 gene directly causes disease progression."

    p1 = get_prob(premise, hypothesis_1)
    p2 = get_prob(premise, hypothesis_2)

    print(f"Entailment prob for explicit negation (Expected: LOW): {p1:.4f}")
    print(f"Entailment prob for causal escalation (Expected: HIGH): {p2:.4f}")

    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

# --- Execute Both Tests ---
try:
    # This assumes 'eval_df' is still in your Colab memory from the main pipeline
    run_discrete_classification_alignment(eval_df)
    run_nli_lexical_resilience()
except NameError:
    print("Error: 'eval_df' is not in memory. Please run your main pipeline cell first to load the dataset.")

--- Running Discrete Classification Test ---
Loading zero-shot classifier (facebook/bart-large-mnli)...
Discrete Classification Failures on Escalations (First 10 pairs): 5/10

--- Running NLI Lexical Resilience Test ---
Loading Asymmetrical NLI Model (cross-encoder/nli-deberta-v3-large)...
Entailment prob for explicit negation (Expected: LOW): 0.0000
Entailment prob for causal escalation (Expected: HIGH): 0.0003


In [26]:
import pandas as pd

def extract_error_analysis():
    print("Loading data for qualitative error analysis...")

    # FIX 1: Point to the new LLM results and the correct dataset path
    try:
        results_df = pd.read_csv("results/esm_evaluation_results_llm.csv")
        # Ensure this matches the exact path used in Cell 4
        text_df = pd.read_csv("data/custom_epistemic_suite.csv")
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Please ensure you are in the correct Drive directory.")
        return

    orig_texts = text_df[text_df["condition"] == "original"].set_index("pair_id")["text"].to_dict()

    def get_hypothesis(row):
        matches = text_df[(text_df["pair_id"] == row["pair_id"]) & (text_df["condition"] == row["test_condition"])]["text"].values
        return matches[0] if len(matches) > 0 else "TEXT NOT FOUND"

    results_df["reference_text"] = results_df["pair_id"].map(orig_texts)
    results_df["hypothesis_text"] = results_df.apply(get_hypothesis, axis=1)

    # --- 1. BERTScore Error Analysis ---
    print("\n" + "="*80)
    print(" BERTScore ALGORITHMIC BLIND SPOTS (Top 3 Errors)")
    print("="*80)

    print("\n--- FALSE NEGATIVES (Escalations missed by BERTScore) ---")
    fn_bert = results_df[results_df["ground_truth_label"] == 1].sort_values("score_bertscore", ascending=False).head(3)
    for _, row in fn_bert.iterrows():
        print(f"\n[Pair: {row['pair_id']} | BERTScore: {row['score_bertscore']:.4f}]")
        print(f"Original : {row['reference_text']}")
        print(f"Escalated: {row['hypothesis_text']}")

    print("\n--- FALSE POSITIVES (Paraphrases unfairly penalized by BERTScore) ---")
    fp_bert = results_df[results_df["ground_truth_label"] == 0].sort_values("score_bertscore", ascending=True).head(3)
    for _, row in fp_bert.iterrows():
        print(f"\n[Pair: {row['pair_id']} | BERTScore: {row['score_bertscore']:.4f}]")
        print(f"Original : {row['reference_text']}")
        print(f"Paraphrase: {row['hypothesis_text']}")

    # --- 2. Generative Δ_epistemic Analysis ---
    print("\n" + "="*80)
    print(" GENERATIVE Δ_EPISTEMIC LOGIC COLLAPSE (Top 3 Extremes)")
    print("="*80)

    # FIX 2: Target the correct LLM score column (score_delta_epistemic_llm)
    print("\n--- FALSE NEGATIVES (Severe epistemic shifts ignored by Generative LLM) ---")
    fn_delta = results_df[results_df["ground_truth_label"] == 1].sort_values("score_delta_epistemic_llm", ascending=True).head(3)
    for _, row in fn_delta.iterrows():
        print(f"\n[Pair: {row['pair_id']} | Δ_epistemic_LLM: {row['score_delta_epistemic_llm']:.4f}]")
        print(f"Original : {row['reference_text']}")
        print(f"Escalated: {row['hypothesis_text']}")

    print("\n--- FALSE POSITIVES (Safe paraphrases flagged as escalations by Generative LLM) ---")
    fp_delta = results_df[results_df["ground_truth_label"] == 0].sort_values("score_delta_epistemic_llm", ascending=False).head(3)
    for _, row in fp_delta.iterrows():
        print(f"\n[Pair: {row['pair_id']} | Δ_epistemic_LLM: {row['score_delta_epistemic_llm']:.4f}]")
        print(f"Original : {row['reference_text']}")
        print(f"Paraphrase: {row['hypothesis_text']}")

extract_error_analysis()

Loading data for qualitative error analysis...

 BERTScore ALGORITHMIC BLIND SPOTS (Top 3 Errors)

--- FALSE NEGATIVES (Escalations missed by BERTScore) ---

[Pair: P072 | BERTScore: 0.9984]
Original : The study population comprised 29 554 men with an ID and 518 739 comparators from the Clinical Practice Research Datalink Aurum database, which is linked to hospital, mortality, and cancer registry data.
Escalated: The study population comprised 29 554 men with an ID and 518 739 comparators from the Clinical Practice Research Datalink Aurum database, which is connected to hospital, mortality, and cancer registry data.

[Pair: P083 | BERTScore: 0.9892]
Original : In both cohorts, it was associated with an increased risk of exacerbations: OR (95% CI) 1.58 (1.07-2.32), <i>p</i> = .02, and 2.20 (1.38-3.52), <i>p</i> < .001, respectively.
Escalated: In both cohorts, it contributed to an increased risk of exacerbations: OR (95% CI) 1.58 (1.07-2.32), <i>p</i> = .02, and 2.20 (1.38-3.52), <i>p</